# Transfer Learning — Reutilizar Conocimiento entre Tareas

## Notebook educativo — Implementación desde cero + ejemplos con librerías

### Objetivo
Comprender y construir desde cero los fundamentos de **Transfer Learning**: cómo reutilizar el conocimiento aprendido en una tarea fuente para mejorar el rendimiento en una tarea objetivo con pocos datos.

### Contenido
1. **¿Qué es Transfer Learning?** — Motivación, taxonomía y cuándo funciona
2. **Domain Shift y medidas de distancia** — Discrepancia entre dominios
3. **Feature Extraction** — Usar representaciones pre-entrenadas como features fijas
4. **Fine-Tuning** — Adaptar capas pre-entrenadas a una nueva tarea
5. **Congelamiento progresivo (Gradual Unfreezing)** — Descongelar capas estratégicamente
6. **Domain Adaptation** — Alinear distribuciones de dominios diferentes
7. **Knowledge Distillation** — Transferir conocimiento de modelo grande a pequeño
8. **Multi-Task Learning** — Aprender múltiples tareas simultáneamente
9. **Transfer Learning Negativo** — Cuándo transferir empeora el resultado
10. **Experimento completo** — Pipeline end-to-end con comparación de estrategias

### Requisitos
```
numpy matplotlib scikit-learn
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_digits, make_moons, make_circles
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from copy import deepcopy
import warnings
import time
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
np.random.seed(42)
print("✅ Entorno listo")

## 1. ¿Qué es Transfer Learning?

### El problema
Entrenar modelos de deep learning desde cero requiere **enormes cantidades de datos y cómputo**. Pero en la práctica:
- Los hospitales tienen pocas radiografías etiquetadas de una enfermedad rara
- Una startup no tiene millones de imágenes para entrenar un detector
- Un nuevo idioma tiene poco texto etiquetado para NLP

### La solución: Transfer Learning

```
Tarea Fuente (muchos datos)          Tarea Objetivo (pocos datos)
┌─────────────────────┐              ┌─────────────────────┐
│ ImageNet (14M imgs)  │              │ Rayos X (500 imgs)   │
│                     │              │                     │
│ Capa 1: bordes      │──transfer──→│ Capa 1: bordes ✓    │
│ Capa 2: texturas    │──transfer──→│ Capa 2: texturas ✓  │
│ Capa 3: partes      │──transfer──→│ Capa 3: partes ✓    │
│ Capa 4: objetos     │──transfer──→│ Capa 4: adaptar     │
│ Capa 5: 1000 clases │     ✗      │ Capa 5: 2 clases    │
└─────────────────────┘              └─────────────────────┘
```

### Taxonomía

| Tipo | Dominio fuente/objetivo | Tarea fuente/objetivo | Ejemplo |
|------|------------------------|----------------------|---------|
| **Inductive TL** | Mismo | Diferente | ImageNet → detección médica |
| **Transductive TL** | Diferente | Misma | Inglés → Español (NLP) |
| **Domain Adaptation** | Diferente | Misma | Fotos → Pinturas |
| **Self-supervised** | Mismo | Pretext → downstream | BERT (MLM → clasificación) |

### ¿Cuándo funciona?
1. La tarea fuente y objetivo comparten **estructura subyacente**
2. Las **capas inferiores** aprenden features genéricas (bordes, texturas, n-gramas)
3. Hay **suficientes datos** en la tarea fuente
4. El dominio no es **demasiado diferente**

### Cuándo NO funciona (Transfer Negativo)
- Dominios completamente diferentes (texto → audio sin relación)
- Tarea fuente demasiado simple para la objetivo
- Demasiado fine-tuning con pocos datos → overfitting

In [ ]:
# ============================================================
# Red Neuronal base para todos los experimentos
# ============================================================

class RedNeuronal:
    """MLP con soporte para transfer learning.
    
    Características:
    - Capas modulares (se pueden congelar individualmente)
    - Soporte para extraer features intermedias
    - Fine-tuning selectivo por capa
    """
    
    def __init__(self, dims, activacion='relu'):
        """dims = [input, hidden1, hidden2, ..., output]"""
        self.dims = dims
        self.n_capas = len(dims) - 1
        self.activacion = activacion
        
        # Inicializar pesos (He initialization)
        self.pesos = []
        self.biases = []
        for i in range(self.n_capas):
            std = np.sqrt(2.0 / dims[i])
            W = np.random.randn(dims[i], dims[i+1]) * std
            b = np.zeros(dims[i+1])
            self.pesos.append(W)
            self.biases.append(b)
        
        # Control de congelamiento por capa
        self.capas_congeladas = [False] * self.n_capas
    
    def _activar(self, z, ultima_capa=False):
        if ultima_capa:
            # Softmax
            exp_z = np.exp(z - z.max(axis=1, keepdims=True))
            return exp_z / exp_z.sum(axis=1, keepdims=True)
        if self.activacion == 'relu':
            return np.maximum(0, z)
        elif self.activacion == 'tanh':
            return np.tanh(z)
    
    def _derivada_activacion(self, h):
        if self.activacion == 'relu':
            return (h > 0).astype(float)
        elif self.activacion == 'tanh':
            return 1 - h ** 2
    
    def forward(self, X, hasta_capa=None):
        """Forward pass. Si hasta_capa != None, retorna features de esa capa."""
        self.activaciones = [X]
        self.zs = []
        h = X
        
        n = hasta_capa if hasta_capa is not None else self.n_capas
        
        for i in range(n):
            z = h @ self.pesos[i] + self.biases[i]
            self.zs.append(z)
            h = self._activar(z, ultima_capa=(i == self.n_capas - 1 and hasta_capa is None))
            self.activaciones.append(h)
        
        return h
    
    def extraer_features(self, X, capa=-1):
        """Extrae features de una capa intermedia (feature extraction)."""
        if capa == -1:
            capa = self.n_capas - 1  # Penúltima capa
        return self.forward(X, hasta_capa=capa)
    
    def backward(self, X, y, lr=0.01):
        """Backward pass con soporte para capas congeladas."""
        n = len(y)
        n_clases = self.pesos[-1].shape[1]
        
        # Gradiente de la capa de salida (softmax + cross-entropy)
        y_onehot = np.zeros((n, n_clases))
        y_onehot[np.arange(n), y] = 1
        delta = self.activaciones[-1] - y_onehot
        delta /= n
        
        # Backpropagation
        for i in range(self.n_capas - 1, -1, -1):
            dW = self.activaciones[i].T @ delta
            db = delta.sum(axis=0)
            
            if i > 0:
                delta = (delta @ self.pesos[i].T) * self._derivada_activacion(self.activaciones[i])
            
            # Solo actualizar si la capa NO está congelada
            if not self.capas_congeladas[i]:
                self.pesos[i] -= lr * dW
                self.biases[i] -= lr * db
    
    def entrenar(self, X, y, epochs=100, lr=0.01, batch_size=32, verbose=False):
        """Entrenamiento por mini-batches."""
        losses = []
        n = len(y)
        
        for epoch in range(epochs):
            idx = np.random.permutation(n)
            epoch_loss = 0
            
            for start in range(0, n, batch_size):
                batch_idx = idx[start:start+batch_size]
                X_batch = X[batch_idx]
                y_batch = y[batch_idx]
                
                # Forward
                probs = self.forward(X_batch)
                loss = -np.mean(np.log(probs[np.arange(len(y_batch)), y_batch] + 1e-10))
                epoch_loss += loss
                
                # Backward
                self.backward(X_batch, y_batch, lr=lr)
            
            losses.append(epoch_loss / (n // batch_size))
            
            if verbose and (epoch + 1) % 50 == 0:
                acc = self.accuracy(X, y)
                print(f"  Epoch {epoch+1}: loss={losses[-1]:.4f}, acc={acc:.4f}")
        
        return losses
    
    def predecir(self, X):
        probs = self.forward(X)
        return np.argmax(probs, axis=1)
    
    def accuracy(self, X, y):
        return np.mean(self.predecir(X) == y)
    
    def congelar_capas(self, capas_indices):
        """Congela capas específicas (no se actualizan)."""
        for i in capas_indices:
            self.capas_congeladas[i] = True
    
    def descongelar_todas(self):
        self.capas_congeladas = [False] * self.n_capas
    
    def copiar_pesos_de(self, otra_red, hasta_capa=None):
        """Copia pesos de otra red (transfer)."""
        n = hasta_capa if hasta_capa is not None else min(self.n_capas, otra_red.n_capas)
        for i in range(n):
            if self.pesos[i].shape == otra_red.pesos[i].shape:
                self.pesos[i] = otra_red.pesos[i].copy()
                self.biases[i] = otra_red.biases[i].copy()
    
    def get_estado(self):
        return [(w.copy(), b.copy()) for w, b in zip(self.pesos, self.biases)]
    
    def set_estado(self, estado):
        for i, (w, b) in enumerate(estado):
            self.pesos[i] = w.copy()
            self.biases[i] = b.copy()

print("✅ Clase RedNeuronal definida con soporte para transfer learning")
print("   - congelar_capas(), copiar_pesos_de(), extraer_features()")

## 2. Domain Shift — ¿Qué tan diferentes son los dominios?

### El problema del Domain Shift
Cuando la distribución de los datos fuente $P_S(X, Y)$ difiere de la distribución objetivo $P_T(X, Y)$:

$$P_S(X, Y) \neq P_T(X, Y)$$

Esto puede descomponerse en:
- **Covariate shift**: $P_S(X) \neq P_T(X)$ pero $P(Y|X)$ es igual
- **Label shift**: $P_S(Y) \neq P_T(Y)$ pero $P(X|Y)$ es igual
- **Concept drift**: $P_S(Y|X) \neq P_T(Y|X)$ — la relación input-output cambia

### Medir la distancia entre dominios

| Medida | Fórmula | Interpretación |
|--------|---------|---------------|
| **MMD** | $\|\mu_S - \mu_T\|_{\mathcal{H}}$ | Distancia de medias en espacio de kernel |
| **KL Divergence** | $\sum P_S \log(P_S/P_T)$ | Divergencia entre distribuciones |
| **A-distance** | $2(1 - 2\epsilon_d)$ | Basada en discriminador de dominio |
| **Wasserstein** | $\inf \mathbb{E}[\|x-y\|]$ | "Costo de transporte" entre distribuciones |

In [ ]:
# ============================================================
# Visualización de Domain Shift y distancias entre dominios
# ============================================================

def crear_dominio_fuente(n=500):
    """Dominio fuente: clusters bien separados."""
    X, y = make_classification(n_samples=n, n_features=2, n_informative=2,
                                n_redundant=0, n_clusters_per_class=1,
                                class_sep=2.0, random_state=42)
    return X, y

def crear_dominio_objetivo(n=50, shift=0, rotacion=0, ruido=0):
    """Dominio objetivo: similar al fuente pero con shift/rotación/ruido."""
    X, y = make_classification(n_samples=n, n_features=2, n_informative=2,
                                n_redundant=0, n_clusters_per_class=1,
                                class_sep=2.0, random_state=42)
    # Aplicar shift
    X += shift
    
    # Aplicar rotación
    if rotacion != 0:
        theta = np.radians(rotacion)
        R = np.array([[np.cos(theta), -np.sin(theta)],
                       [np.sin(theta), np.cos(theta)]])
        X = X @ R.T
    
    # Agregar ruido
    X += np.random.randn(*X.shape) * ruido
    
    return X, y

def mmd_lineal(X_s, X_t):
    """Maximum Mean Discrepancy (kernel lineal): ||mean(X_s) - mean(X_t)||²."""
    return np.sum((X_s.mean(axis=0) - X_t.mean(axis=0)) ** 2)

def mmd_rbf(X_s, X_t, gamma=1.0):
    """MMD con kernel RBF."""
    def kernel(X, Y):
        dists = np.sum(X**2, axis=1, keepdims=True) + np.sum(Y**2, axis=1) - 2 * X @ Y.T
        return np.exp(-gamma * dists)
    
    K_ss = kernel(X_s, X_s).mean()
    K_tt = kernel(X_t, X_t).mean()
    K_st = kernel(X_s, X_t).mean()
    return K_ss + K_tt - 2 * K_st

def proxy_a_distance(X_s, X_t):
    """Proxy A-distance: entrenar discriminador para distinguir dominios."""
    X = np.vstack([X_s, X_t])
    y = np.array([0]*len(X_s) + [1]*len(X_t))
    clf = LogisticRegression(max_iter=500)
    acc = cross_val_score(clf, X, y, cv=3, scoring='accuracy').mean()
    return 2 * (1 - 2 * max(0, acc - 0.5))

# --- Crear escenarios de domain shift ---
X_source, y_source = crear_dominio_fuente(500)

escenarios = {
    'Sin shift': {'shift': 0, 'rotacion': 0, 'ruido': 0},
    'Shift leve': {'shift': 1, 'rotacion': 0, 'ruido': 0.3},
    'Rotación 30°': {'shift': 0, 'rotacion': 30, 'ruido': 0.2},
    'Shift + rotación': {'shift': 2, 'rotacion': 45, 'ruido': 0.5},
    'Mucho ruido': {'shift': 0.5, 'rotacion': 15, 'ruido': 1.5},
    'Domain shift severo': {'shift': 3, 'rotacion': 60, 'ruido': 1.0},
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, (nombre, params) in zip(axes.flat, escenarios.items()):
    X_target, y_target = crear_dominio_objetivo(100, **params)
    
    # Visualizar
    ax.scatter(X_source[y_source==0, 0], X_source[y_source==0, 1], 
              c='lightblue', s=15, alpha=0.3, label='Fuente C0')
    ax.scatter(X_source[y_source==1, 0], X_source[y_source==1, 1], 
              c='lightsalmon', s=15, alpha=0.3, label='Fuente C1')
    ax.scatter(X_target[y_target==0, 0], X_target[y_target==0, 1], 
              c='blue', s=40, marker='x', label='Objetivo C0')
    ax.scatter(X_target[y_target==1, 0], X_target[y_target==1, 1], 
              c='red', s=40, marker='x', label='Objetivo C1')
    
    # Calcular distancias
    mmd_val = mmd_rbf(X_source, X_target, gamma=0.5)
    pad = proxy_a_distance(X_source, X_target)
    
    ax.set_title(f'{nombre}\nMMD={mmd_val:.3f}, PAD={pad:.3f}', fontweight='bold')
    ax.grid(True, alpha=0.3)
    if nombre == 'Sin shift':
        ax.legend(fontsize=7, loc='upper left')

plt.suptitle('Domain Shift: Distribución fuente vs objetivo', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()
print("✅ Mayor MMD/PAD → mayor discrepancia entre dominios → transfer más difícil")

## 3. Feature Extraction — Representaciones pre-entrenadas

### Idea
Usar una red pre-entrenada como **extractor de features fijo** y entrenar solo un clasificador nuevo encima:

```
Red pre-entrenada (congelada)          Clasificador nuevo
┌──────────────────────┐              ┌──────────────┐
│ Input                │              │              │
│   ↓                  │              │              │
│ Capa 1 (congelada) ❄│              │              │
│   ↓                  │──features──→│ Capa nueva   │
│ Capa 2 (congelada) ❄│              │ (entrenable) │
│   ↓                  │              │              │
│ Capa 3 (congelada) ❄│              └──────────────┘
└──────────────────────┘
```

### Ventajas
- **Rápido**: solo se entrena el clasificador final
- **Pocos datos**: las features genéricas ya están aprendidas
- **Sin overfitting**: menos parámetros que entrenar

### Cuándo usarlo
- Dataset objetivo **pequeño** y **similar** al fuente
- No se necesita mucha adaptación de las features

In [ ]:
# ============================================================
# Feature Extraction desde cero
# ============================================================

def crear_tarea_fuente(n=2000):
    """Tarea fuente: clasificación con muchos datos (10 clases)."""
    X, y = make_classification(n_samples=n, n_features=20, n_informative=15,
                                n_redundant=3, n_classes=10, n_clusters_per_class=1,
                                random_state=42)
    return StandardScaler().fit_transform(X), y

def crear_tarea_objetivo(n=100, dificultad='facil'):
    """Tarea objetivo: clasificación con pocos datos (3 clases)."""
    if dificultad == 'facil':
        X, y = make_classification(n_samples=n, n_features=20, n_informative=15,
                                    n_redundant=3, n_classes=3, class_sep=2.0,
                                    random_state=123)
    elif dificultad == 'media':
        X, y = make_classification(n_samples=n, n_features=20, n_informative=10,
                                    n_redundant=5, n_classes=3, class_sep=1.0,
                                    random_state=123)
    else:
        X, y = make_classification(n_samples=n, n_features=20, n_informative=6,
                                    n_redundant=8, n_classes=3, class_sep=0.5,
                                    flip_y=0.1, random_state=123)
    return StandardScaler().fit_transform(X), y

# --- Paso 1: Entrenar red en tarea fuente (muchos datos) ---
print("=== PASO 1: Pre-entrenar en tarea fuente (2000 ejemplos, 10 clases) ===\n")
X_fuente, y_fuente = crear_tarea_fuente(2000)
X_f_train, X_f_test, y_f_train, y_f_test = train_test_split(
    X_fuente, y_fuente, test_size=0.2, random_state=42)

red_fuente = RedNeuronal([20, 128, 64, 32, 10])
losses_fuente = red_fuente.entrenar(X_f_train, y_f_train, epochs=200, lr=0.01, verbose=True)
acc_fuente = red_fuente.accuracy(X_f_test, y_f_test)
print(f"\nAccuracy en tarea fuente: {acc_fuente:.4f}")

# --- Paso 2: Feature Extraction para tarea objetivo ---
print("\n=== PASO 2: Feature Extraction para tarea objetivo (100 ejemplos, 3 clases) ===\n")
X_obj, y_obj = crear_tarea_objetivo(100, 'media')
X_o_train, X_o_test, y_o_train, y_o_test = train_test_split(
    X_obj, y_obj, test_size=0.3, random_state=42)

# Estrategia A: Entrenar desde cero
red_scratch = RedNeuronal([20, 128, 64, 32, 3])
losses_scratch = red_scratch.entrenar(X_o_train, y_o_train, epochs=200, lr=0.01)
acc_scratch = red_scratch.accuracy(X_o_test, y_o_test)
print(f"Desde cero:        {acc_scratch:.4f}")

# Estrategia B: Feature Extraction (capas congeladas + clasificador nuevo)
# Extraer features de la capa 3 (antes de la capa de salida)
features_train = red_fuente.extraer_features(X_o_train, capa=3)  # dim=32
features_test = red_fuente.extraer_features(X_o_test, capa=3)

# Clasificador simple sobre las features extraídas
clf_fe = LogisticRegression(max_iter=500)
clf_fe.fit(features_train, y_o_train)
acc_fe = clf_fe.score(features_test, y_o_test)
print(f"Feature Extraction: {acc_fe:.4f}")

# Estrategia B2: Feature Extraction con red nueva
red_fe = RedNeuronal([32, 16, 3])  # Pequeña red sobre features
X_fe_train = red_fuente.extraer_features(X_o_train, capa=3)
X_fe_test = red_fuente.extraer_features(X_o_test, capa=3)
losses_fe = red_fe.entrenar(X_fe_train, y_o_train, epochs=200, lr=0.01)
acc_fe2 = red_fe.accuracy(X_fe_test, y_o_test)
print(f"Feature Ext + MLP: {acc_fe2:.4f}")

# Estrategia C: Solo baseline
clf_base = LogisticRegression(max_iter=500).fit(X_o_train, y_o_train)
acc_base = clf_base.score(X_o_test, y_o_test)
print(f"LogReg baseline:   {acc_base:.4f}")

print(f"\n→ Feature Extraction reutiliza representaciones aprendidas con muchos datos")

## 4. Fine-Tuning — Adaptar la red pre-entrenada

### Idea
En vez de congelar toda la red, **adaptar** los pesos pre-entrenados a la nueva tarea:

### Estrategias de Fine-Tuning

| Estrategia | Capas entrenables | Cuándo usar |
|-----------|-------------------|-------------|
| **Feature Extraction** | Solo clasificador final | Muy pocos datos, dominios similares |
| **Fine-tune última capa** | Última(s) capa(s) + clasificador | Pocos datos, dominios similares |
| **Fine-tune completo** | Todas las capas (LR bajo) | Datos moderados, dominios diferentes |
| **Gradual unfreezing** | Descongelar progresivamente | Mejor práctica general |

### Learning Rate para Fine-Tuning
Es crucial usar un **learning rate más bajo** que el entrenamiento original:
- LR original: $10^{-3}$
- LR fine-tuning: $10^{-4}$ a $10^{-5}$

Si el LR es muy alto, se "destruyen" las features pre-entrenadas (**catastrophic forgetting**).

### Discriminative Learning Rates (Howard & Ruder, 2018)
Usar LR **diferentes por capa**: más bajo en capas inferiores, más alto en capas superiores:

$$\eta_l = \eta_{base} \cdot \alpha^{L-l}$$

donde $\alpha < 1$ es el factor de decaimiento y $l$ es el índice de capa.

In [ ]:
# ============================================================
# Fine-Tuning: múltiples estrategias desde cero
# ============================================================

class TransferLearner:
    """Sistema de Transfer Learning con múltiples estrategias."""
    
    def __init__(self, red_fuente, dims_objetivo):
        self.red_fuente = red_fuente
        self.dims_objetivo = dims_objetivo
    
    def feature_extraction(self, X_train, y_train, X_test, y_test, 
                           capa_features=3, epochs=200, lr=0.01):
        """Congela toda la red, entrena solo clasificador nuevo."""
        feat_train = self.red_fuente.extraer_features(X_train, capa=capa_features)
        feat_test = self.red_fuente.extraer_features(X_test, capa=capa_features)
        
        feat_dim = feat_train.shape[1]
        n_clases = len(np.unique(y_train))
        clasificador = RedNeuronal([feat_dim, 16, n_clases])
        losses = clasificador.entrenar(feat_train, y_train, epochs=epochs, lr=lr)
        acc = clasificador.accuracy(feat_test, y_test)
        return acc, losses
    
    def fine_tune_ultima_capa(self, X_train, y_train, X_test, y_test,
                               epochs=200, lr=0.001):
        """Congela todo excepto la última capa oculta y la de salida."""
        n_clases = len(np.unique(y_train))
        dims = list(self.red_fuente.dims[:-1]) + [n_clases]
        
        red_ft = RedNeuronal(dims)
        # Copiar pesos pre-entrenados
        for i in range(len(self.red_fuente.pesos) - 1):
            red_ft.pesos[i] = self.red_fuente.pesos[i].copy()
            red_ft.biases[i] = self.red_fuente.biases[i].copy()
        
        # Congelar todas menos las últimas 2 capas
        for i in range(len(red_ft.pesos) - 2):
            red_ft.capas_congeladas[i] = True
        
        losses = red_ft.entrenar(X_train, y_train, epochs=epochs, lr=lr)
        acc = red_ft.accuracy(X_test, y_test)
        return acc, losses
    
    def fine_tune_completo(self, X_train, y_train, X_test, y_test,
                           epochs=200, lr=0.0005):
        """Fine-tune todas las capas con LR bajo."""
        n_clases = len(np.unique(y_train))
        dims = list(self.red_fuente.dims[:-1]) + [n_clases]
        
        red_ft = RedNeuronal(dims)
        for i in range(len(self.red_fuente.pesos) - 1):
            red_ft.pesos[i] = self.red_fuente.pesos[i].copy()
            red_ft.biases[i] = self.red_fuente.biases[i].copy()
        
        losses = red_ft.entrenar(X_train, y_train, epochs=epochs, lr=lr)
        acc = red_ft.accuracy(X_test, y_test)
        return acc, losses
    
    def gradual_unfreezing(self, X_train, y_train, X_test, y_test,
                           epochs_por_fase=50, lr_base=0.001, decay=0.5):
        """Descongelar capas progresivamente (Howard & Ruder, 2018)."""
        n_clases = len(np.unique(y_train))
        dims = list(self.red_fuente.dims[:-1]) + [n_clases]
        
        red_ft = RedNeuronal(dims)
        for i in range(len(self.red_fuente.pesos) - 1):
            red_ft.pesos[i] = self.red_fuente.pesos[i].copy()
            red_ft.biases[i] = self.red_fuente.biases[i].copy()
        
        # Congelar todas
        red_ft.congelar_capas(list(range(red_ft.n_capas)))
        
        all_losses = []
        accs_por_fase = []
        
        # Descongelar de arriba hacia abajo
        for fase in range(red_ft.n_capas):
            capa_idx = red_ft.n_capas - 1 - fase
            red_ft.capas_congeladas[capa_idx] = False
            
            lr = lr_base * (decay ** fase)
            losses = red_ft.entrenar(X_train, y_train, epochs=epochs_por_fase, lr=lr)
            all_losses.extend(losses)
            
            acc = red_ft.accuracy(X_test, y_test)
            accs_por_fase.append(acc)
        
        return accs_por_fase[-1], all_losses, accs_por_fase
    
    def discriminative_lr(self, X_train, y_train, X_test, y_test,
                          epochs=200, lr_top=0.005, decay=0.3):
        """Fine-tune con learning rates discriminativos por capa."""
        n_clases = len(np.unique(y_train))
        dims = list(self.red_fuente.dims[:-1]) + [n_clases]
        
        red_ft = RedNeuronal(dims)
        for i in range(len(self.red_fuente.pesos) - 1):
            red_ft.pesos[i] = self.red_fuente.pesos[i].copy()
            red_ft.biases[i] = self.red_fuente.biases[i].copy()
        
        losses = []
        n = len(y_train)
        n_clases_out = n_clases
        
        for epoch in range(epochs):
            idx = np.random.permutation(n)
            epoch_loss = 0
            
            for start in range(0, n, 32):
                batch_idx = idx[start:start+32]
                X_b = X_train[batch_idx]
                y_b = y_train[batch_idx]
                
                probs = red_ft.forward(X_b)
                loss = -np.mean(np.log(probs[np.arange(len(y_b)), y_b] + 1e-10))
                epoch_loss += loss
                
                # Backward manual con LR por capa
                y_oh = np.zeros((len(y_b), n_clases_out))
                y_oh[np.arange(len(y_b)), y_b] = 1
                delta = (probs - y_oh) / len(y_b)
                
                for i in range(red_ft.n_capas - 1, -1, -1):
                    dW = red_ft.activaciones[i].T @ delta
                    db = delta.sum(axis=0)
                    
                    if i > 0:
                        delta = (delta @ red_ft.pesos[i].T) * red_ft._derivada_activacion(red_ft.activaciones[i])
                    
                    # LR discriminativo: más bajo para capas inferiores
                    lr_capa = lr_top * (decay ** (red_ft.n_capas - 1 - i))
                    red_ft.pesos[i] -= lr_capa * dW
                    red_ft.biases[i] -= lr_capa * db
            
            losses.append(epoch_loss / (n // 32))
        
        acc = red_ft.accuracy(X_test, y_test)
        return acc, losses

# --- Comparar todas las estrategias ---
print("=== Comparación de Estrategias de Transfer Learning ===\n")

transfer = TransferLearner(red_fuente, dims_objetivo=[20, 128, 64, 32, 3])

estrategias = {}

# Desde cero (sin transfer)
red_scratch = RedNeuronal([20, 128, 64, 32, 3])
losses_s = red_scratch.entrenar(X_o_train, y_o_train, epochs=200, lr=0.01)
estrategias['Desde cero'] = (red_scratch.accuracy(X_o_test, y_o_test), losses_s)

# Feature Extraction
acc, losses = transfer.feature_extraction(X_o_train, y_o_train, X_o_test, y_o_test)
estrategias['Feature Extraction'] = (acc, losses)

# Fine-tune última capa
acc, losses = transfer.fine_tune_ultima_capa(X_o_train, y_o_train, X_o_test, y_o_test)
estrategias['Fine-tune última'] = (acc, losses)

# Fine-tune completo
acc, losses = transfer.fine_tune_completo(X_o_train, y_o_train, X_o_test, y_o_test)
estrategias['Fine-tune completo'] = (acc, losses)

# Gradual unfreezing
acc, losses, accs_fases = transfer.gradual_unfreezing(X_o_train, y_o_train, X_o_test, y_o_test)
estrategias['Gradual Unfreezing'] = (acc, losses)

# Discriminative LR
acc, losses = transfer.discriminative_lr(X_o_train, y_o_train, X_o_test, y_o_test)
estrategias['Discriminative LR'] = (acc, losses)

for nombre, (acc, _) in estrategias.items():
    bar = '█' * int(acc * 40)
    print(f"  {nombre:25s}: {acc:.4f}  {bar}")

# Visualizar
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Barras de accuracy
ax = axes[0]
nombres = list(estrategias.keys())
accs = [estrategias[n][0] for n in nombres]
colores = ['lightcoral', 'skyblue', 'steelblue', 'royalblue', 'navy', 'purple']
ax.barh(range(len(nombres)), accs, color=colores, alpha=0.8, edgecolor='black')
ax.set_yticks(range(len(nombres))); ax.set_yticklabels(nombres, fontsize=10)
ax.set_xlabel('Accuracy'); ax.set_title('Comparación de Estrategias', fontweight='bold')
ax.invert_yaxis()

# 2. Curvas de loss
ax = axes[1]
for nombre, (_, losses) in estrategias.items():
    if losses:
        ax.plot(losses[:200], lw=1.5, alpha=0.8, label=nombre)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Curvas de Entrenamiento', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# 3. Gradual Unfreezing: accuracy por fase
ax = axes[2]
fases_nombres = [f'Descongelar\ncapa {4-i}' for i in range(len(accs_fases))]
ax.bar(range(len(accs_fases)), accs_fases, color=['#e74c3c', '#e67e22', '#2ecc71', '#3498db'],
       alpha=0.8, edgecolor='black')
ax.set_xticks(range(len(fases_nombres))); ax.set_xticklabels(fases_nombres, fontsize=9)
ax.set_ylabel('Accuracy'); ax.set_title('Gradual Unfreezing\nAccuracy por fase', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout(); plt.show()

## 5. Domain Adaptation — Alinear distribuciones

### El problema
Cuando los datos fuente y objetivo tienen **distribuciones diferentes**, incluso buenos features pre-entrenados pueden fallar.

### Soluciones

**1. Feature Alignment (CORAL — Sun et al., 2016)**
Alinear las estadísticas de segundo orden (covarianza) de los features:

$$\hat{X}_S = X_S \cdot C_S^{-1/2} \cdot C_T^{1/2}$$

**2. Domain-Adversarial Training (DANN — Ganin et al., 2016)**
Entrenar un extractor de features que sea:
- **Bueno** para clasificar la tarea (task loss)
- **Malo** para distinguir el dominio (domain loss → gradient reversal)

```
          Feature Extractor
                │
        ┌───────┴───────┐
        │               │
   Task Head      Domain Head
   (clasificar)   (distinguir dominio)
        ↓               ↓
   Minimizar       Maximizar error
   task loss       domain loss
```

**3. Instance Reweighting**
Dar más peso a ejemplos fuente que son más similares al dominio objetivo.

In [ ]:
# ============================================================
# Domain Adaptation desde cero
# ============================================================

class CORAL:
    """CORrelation ALignment: alinear covarianzas entre dominios."""
    
    def __init__(self):
        self.C_s = None
        self.C_t = None
        self.media_s = None
        self.media_t = None
    
    def fit(self, X_source, X_target):
        """Calcula las matrices de covarianza de ambos dominios."""
        self.media_s = X_source.mean(axis=0)
        self.media_t = X_target.mean(axis=0)
        
        X_s_c = X_source - self.media_s
        X_t_c = X_target - self.media_t
        
        self.C_s = (X_s_c.T @ X_s_c) / (len(X_source) - 1) + 1e-6 * np.eye(X_source.shape[1])
        self.C_t = (X_t_c.T @ X_t_c) / (len(X_target) - 1) + 1e-6 * np.eye(X_target.shape[1])
        return self
    
    def transformar_source(self, X_source):
        """Transforma features fuente para alinear con distribución objetivo."""
        X_c = X_source - self.media_s
        
        # C_s^{-1/2}: raíz cuadrada inversa de C_s
        eigvals_s, eigvecs_s = np.linalg.eigh(self.C_s)
        C_s_inv_sqrt = eigvecs_s @ np.diag(1.0 / np.sqrt(np.maximum(eigvals_s, 1e-10))) @ eigvecs_s.T
        
        # C_t^{1/2}: raíz cuadrada de C_t
        eigvals_t, eigvecs_t = np.linalg.eigh(self.C_t)
        C_t_sqrt = eigvecs_t @ np.diag(np.sqrt(np.maximum(eigvals_t, 1e-10))) @ eigvecs_t.T
        
        X_aligned = X_c @ C_s_inv_sqrt @ C_t_sqrt + self.media_t
        return X_aligned


class DomainAdversarial:
    """Domain-Adversarial Neural Network (DANN) simplificado."""
    
    def __init__(self, input_dim, feature_dim=32, n_clases=2):
        # Feature extractor
        self.W_feat = np.random.randn(input_dim, feature_dim) * np.sqrt(2/input_dim)
        self.b_feat = np.zeros(feature_dim)
        
        # Task classifier
        self.W_task = np.random.randn(feature_dim, n_clases) * np.sqrt(2/feature_dim)
        self.b_task = np.zeros(n_clases)
        
        # Domain discriminator
        self.W_domain = np.random.randn(feature_dim, 2) * np.sqrt(2/feature_dim)
        self.b_domain = np.zeros(2)
    
    def _relu(self, x):
        return np.maximum(0, x)
    
    def _softmax(self, x):
        exp_x = np.exp(x - x.max(axis=1, keepdims=True))
        return exp_x / exp_x.sum(axis=1, keepdims=True)
    
    def entrenar(self, X_source, y_source, X_target, epochs=200, lr=0.01, 
                 lambda_domain=0.1, verbose=True):
        """Entrenamiento adversarial."""
        if verbose:
            print("=== Domain-Adversarial Training ===\n")
        
        losses_task, losses_domain = [], []
        
        for epoch in range(epochs):
            # Features
            feat_s = self._relu(X_source @ self.W_feat + self.b_feat)
            feat_t = self._relu(X_target @ self.W_feat + self.b_feat)
            
            # === Task loss (solo source, tiene labels) ===
            logits_task = feat_s @ self.W_task + self.b_task
            probs_task = self._softmax(logits_task)
            n_s = len(y_source)
            task_loss = -np.mean(np.log(probs_task[np.arange(n_s), y_source] + 1e-10))
            
            # Gradiente task
            y_oh = np.zeros_like(probs_task)
            y_oh[np.arange(n_s), y_source] = 1
            d_task = (probs_task - y_oh) / n_s
            
            dW_task = feat_s.T @ d_task
            db_task = d_task.sum(axis=0)
            
            # === Domain loss (ambos dominios) ===
            feat_all = np.vstack([feat_s, feat_t])
            domain_labels = np.array([0]*len(X_source) + [1]*len(X_target))
            
            logits_domain = feat_all @ self.W_domain + self.b_domain
            probs_domain = self._softmax(logits_domain)
            n_all = len(domain_labels)
            domain_loss = -np.mean(np.log(probs_domain[np.arange(n_all), domain_labels] + 1e-10))
            
            d_oh = np.zeros_like(probs_domain)
            d_oh[np.arange(n_all), domain_labels] = 1
            d_domain = (probs_domain - d_oh) / n_all
            
            dW_domain = feat_all.T @ d_domain
            db_domain = d_domain.sum(axis=0)
            
            # Gradient reversal: gradiente del domain va NEGADO al feature extractor
            d_feat_from_task = d_task @ self.W_task.T * (feat_s > 0)
            d_feat_from_domain = d_domain @ self.W_domain.T * (np.vstack([feat_s, feat_t]) > 0)
            
            # Combinar: task gradient - lambda * domain gradient (gradient reversal)
            d_feat_source = d_feat_from_task - lambda_domain * d_feat_from_domain[:n_s]
            
            dW_feat = X_source.T @ d_feat_source
            db_feat = d_feat_source.sum(axis=0)
            
            # Actualizar
            self.W_task -= lr * dW_task
            self.b_task -= lr * db_task
            self.W_domain -= lr * dW_domain  # Domain se entrena para discriminar
            self.b_domain -= lr * db_domain
            self.W_feat -= lr * dW_feat       # Features se entrenan para confundir
            self.b_feat -= lr * db_feat
            
            losses_task.append(task_loss)
            losses_domain.append(domain_loss)
            
            if verbose and (epoch + 1) % 50 == 0:
                print(f"  Epoch {epoch+1}: task_loss={task_loss:.4f}, domain_loss={domain_loss:.4f}")
        
        return losses_task, losses_domain
    
    def predecir(self, X):
        feat = self._relu(X @ self.W_feat + self.b_feat)
        logits = feat @ self.W_task + self.b_task
        return np.argmax(logits, axis=1)

# --- Experimento Domain Adaptation ---
print("=== Experimento: Domain Adaptation ===\n")

# Crear dominios con shift
X_s, y_s = crear_dominio_fuente(500)
X_t, y_t = crear_dominio_objetivo(100, shift=2, rotacion=30, ruido=0.5)
X_t_train, X_t_test, y_t_train, y_t_test = train_test_split(X_t, y_t, test_size=0.5, random_state=42)

# 1. Sin adaptación: entrenar en source, evaluar en target
clf_no_adapt = LogisticRegression(max_iter=500).fit(X_s, y_s)
acc_no_adapt = clf_no_adapt.score(X_t_test, y_t_test)
print(f"Sin adaptación:            {acc_no_adapt:.4f}")

# 2. CORAL
coral = CORAL().fit(X_s, X_t_train)
X_s_aligned = coral.transformar_source(X_s)
clf_coral = LogisticRegression(max_iter=500).fit(X_s_aligned, y_s)
acc_coral = clf_coral.score(X_t_test, y_t_test)
print(f"CORAL:                     {acc_coral:.4f}")

# 3. DANN
dann = DomainAdversarial(input_dim=2, feature_dim=16, n_clases=2)
dann.entrenar(X_s, y_s, X_t_train, epochs=200, lr=0.01, lambda_domain=0.3, verbose=False)
acc_dann = accuracy_score(y_t_test, dann.predecir(X_t_test))
print(f"DANN:                      {acc_dann:.4f}")

# 4. Entrenar directamente en target (upper bound con pocos datos)
clf_target = LogisticRegression(max_iter=500).fit(X_t_train, y_t_train)
acc_target = clf_target.score(X_t_test, y_t_test)
print(f"Solo target (upper bound): {acc_target:.4f}")

## 6. Knowledge Distillation — Comprimir modelos

### Idea (Hinton et al., 2015)
Transferir conocimiento de un modelo **grande** (teacher) a uno **pequeño** (student):

$$\mathcal{L}_{\text{distill}} = (1-\alpha) \cdot \mathcal{L}_{CE}(y, p_{student}) + \alpha \cdot T^2 \cdot \text{KL}(\sigma(z_T/T), \sigma(z_S/T))$$

Donde:
- $T$ = temperatura (suaviza las probabilidades)
- $z_T$ = logits del teacher, $z_S$ = logits del student
- $\alpha$ = peso de la pérdida de distilación

### ¿Por qué funciona?
Las **soft labels** del teacher contienen más información que las hard labels:
- Hard label: [0, 0, 1, 0] → solo dice "es clase 2"
- Soft label: [0.01, 0.05, 0.85, 0.09] → dice "es clase 2, algo parecido a 3, nada que ver con 0"

Esta información sobre **relaciones entre clases** es el "dark knowledge".

In [ ]:
# ============================================================
# Knowledge Distillation desde cero
# ============================================================

class KnowledgeDistiller:
    """Transfiere conocimiento de un teacher grande a un student pequeño."""
    
    def __init__(self, teacher, student_dims, temperatura=3.0, alpha=0.7):
        self.teacher = teacher
        self.student = RedNeuronal(student_dims)
        self.T = temperatura
        self.alpha = alpha
    
    def _softmax_temperatura(self, logits, T):
        """Softmax con temperatura: distribución más suave."""
        scaled = logits / T
        exp_s = np.exp(scaled - scaled.max(axis=1, keepdims=True))
        return exp_s / exp_s.sum(axis=1, keepdims=True)
    
    def _kl_divergence(self, p, q):
        """KL(p || q) = Σ p * log(p/q)"""
        return np.sum(p * np.log((p + 1e-10) / (q + 1e-10)), axis=1).mean()
    
    def entrenar(self, X_train, y_train, epochs=200, lr=0.01, verbose=True):
        """Entrenamiento con distilación."""
        if verbose:
            print("=== Knowledge Distillation ===")
            print(f"    Teacher: {self.teacher.dims}")
            print(f"    Student: {self.student.dims}")
            print(f"    T={self.T}, α={self.alpha}\n")
        
        n = len(y_train)
        n_clases = self.teacher.dims[-1]
        losses = []
        
        for epoch in range(epochs):
            idx = np.random.permutation(n)
            epoch_loss = 0
            
            for start in range(0, n, 32):
                batch_idx = idx[start:start+32]
                X_b = X_train[batch_idx]
                y_b = y_train[batch_idx]
                
                # Teacher forward (sin gradientes)
                teacher_probs = self.teacher.forward(X_b)
                # Obtener logits del teacher (pre-softmax)
                teacher_logits = self.teacher.zs[-1]
                teacher_soft = self._softmax_temperatura(teacher_logits, self.T)
                
                # Student forward
                student_probs = self.student.forward(X_b)
                student_logits = self.student.zs[-1]
                student_soft = self._softmax_temperatura(student_logits, self.T)
                
                # Loss = α * T² * KL(teacher_soft, student_soft) + (1-α) * CE(y, student)
                kl_loss = self._kl_divergence(teacher_soft, student_soft)
                
                y_oh = np.zeros((len(y_b), n_clases))
                y_oh[np.arange(len(y_b)), y_b] = 1
                ce_loss = -np.mean(np.sum(y_oh * np.log(student_probs + 1e-10), axis=1))
                
                total_loss = self.alpha * (self.T ** 2) * kl_loss + (1 - self.alpha) * ce_loss
                epoch_loss += total_loss
                
                # Gradiente combinado
                d_soft = self.alpha * (student_soft - teacher_soft) / len(y_b)
                d_hard = (1 - self.alpha) * (student_probs - y_oh) / len(y_b)
                delta = d_soft + d_hard
                
                # Backward del student
                for i in range(self.student.n_capas - 1, -1, -1):
                    dW = self.student.activaciones[i].T @ delta
                    db = delta.sum(axis=0)
                    if i > 0:
                        delta = (delta @ self.student.pesos[i].T) * self.student._derivada_activacion(self.student.activaciones[i])
                    self.student.pesos[i] -= lr * dW
                    self.student.biases[i] -= lr * db
            
            losses.append(epoch_loss / (n // 32))
            
            if verbose and (epoch + 1) % 50 == 0:
                acc = self.student.accuracy(X_train, y_train)
                print(f"  Epoch {epoch+1}: loss={losses[-1]:.4f}, acc={acc:.4f}")
        
        return losses

# --- Experimento: Knowledge Distillation ---
print("Entrenando Teacher (grande)...\n")
X_kd, y_kd = make_classification(n_samples=1000, n_features=20, n_informative=15,
                                   n_classes=5, random_state=42)
X_kd = StandardScaler().fit_transform(X_kd)
X_kd_train, X_kd_test, y_kd_train, y_kd_test = train_test_split(X_kd, y_kd, test_size=0.2)

# Teacher: red grande
teacher = RedNeuronal([20, 256, 128, 64, 5])
teacher.entrenar(X_kd_train, y_kd_train, epochs=300, lr=0.01, verbose=True)
acc_teacher = teacher.accuracy(X_kd_test, y_kd_test)
print(f"\nTeacher accuracy: {acc_teacher:.4f}")
n_params_teacher = sum(w.size + b.size for w, b in zip(teacher.pesos, teacher.biases))

# Student sin distilación
print("\nStudent sin distilación:")
student_solo = RedNeuronal([20, 32, 5])
student_solo.entrenar(X_kd_train, y_kd_train, epochs=300, lr=0.01, verbose=True)
acc_student_solo = student_solo.accuracy(X_kd_test, y_kd_test)
n_params_student = sum(w.size + b.size for w, b in zip(student_solo.pesos, student_solo.biases))

# Student con distilación
print("\nStudent CON distilación:")
distiller = KnowledgeDistiller(teacher, student_dims=[20, 32, 5], temperatura=4.0, alpha=0.7)
losses_kd = distiller.entrenar(X_kd_train, y_kd_train, epochs=300, lr=0.01)
acc_student_kd = distiller.student.accuracy(X_kd_test, y_kd_test)

print(f"\n=== Resultados ===")
print(f"Teacher ({n_params_teacher:,} params):       {acc_teacher:.4f}")
print(f"Student solo ({n_params_student:,} params):   {acc_student_solo:.4f}")
print(f"Student + KD ({n_params_student:,} params):   {acc_student_kd:.4f}")
print(f"\n→ Distilación mejora el student manteniendo el modelo {n_params_teacher/n_params_student:.0f}x más pequeño")

## 7. Multi-Task Learning — Aprender varias tareas a la vez

### Idea
En vez de entrenar modelos separados, compartir representaciones entre tareas relacionadas:

```
          Shared Backbone
                │
        ┌───────┼───────┐
        │       │       │
    Task Head  Task Head Task Head
    (Tarea A)  (Tarea B) (Tarea C)
```

### Tipos
- **Hard sharing**: mismas capas compartidas para todas las tareas
- **Soft sharing**: capas separadas con regularización de similitud

### Ventajas
- **Regularización implícita**: las tareas se regularizan mutuamente
- **Transferencia mutua**: features útiles para una tarea benefician a las otras
- **Eficiencia**: un solo modelo para múltiples tareas

In [ ]:
# ============================================================
# Multi-Task Learning desde cero
# ============================================================

class MultiTaskNet:
    """Red con backbone compartido y heads específicas por tarea."""
    
    def __init__(self, input_dim, shared_dims, task_dims_list):
        """
        input_dim: dimensión de entrada
        shared_dims: [hidden1, hidden2] del backbone compartido
        task_dims_list: lista de [hidden, output] por tarea
        """
        # Backbone compartido
        self.shared_pesos = []
        self.shared_biases = []
        dims = [input_dim] + shared_dims
        for i in range(len(dims) - 1):
            std = np.sqrt(2.0 / dims[i])
            self.shared_pesos.append(np.random.randn(dims[i], dims[i+1]) * std)
            self.shared_biases.append(np.zeros(dims[i+1]))
        
        # Task-specific heads
        self.task_pesos = []
        self.task_biases = []
        for task_dims in task_dims_list:
            pesos, biases = [], []
            dims_t = [shared_dims[-1]] + task_dims
            for i in range(len(dims_t) - 1):
                std = np.sqrt(2.0 / dims_t[i])
                pesos.append(np.random.randn(dims_t[i], dims_t[i+1]) * std)
                biases.append(np.zeros(dims_t[i+1]))
            self.task_pesos.append(pesos)
            self.task_biases.append(biases)
        
        self.n_tasks = len(task_dims_list)
    
    def forward(self, X, task_id):
        """Forward para una tarea específica."""
        # Backbone compartido
        h = X
        self.shared_acts = [h]
        for W, b in zip(self.shared_pesos, self.shared_biases):
            z = h @ W + b
            h = np.maximum(0, z)
            self.shared_acts.append(h)
        
        # Task head
        self.task_acts = [h]
        pesos = self.task_pesos[task_id]
        biases = self.task_biases[task_id]
        for i, (W, b) in enumerate(zip(pesos, biases)):
            z = h @ W + b
            if i < len(pesos) - 1:
                h = np.maximum(0, z)
            else:
                exp_z = np.exp(z - z.max(axis=1, keepdims=True))
                h = exp_z / exp_z.sum(axis=1, keepdims=True)
            self.task_acts.append(h)
        
        return h
    
    def entrenar(self, tareas_datos, epochs=200, lr=0.01, verbose=True):
        """Entrenamiento multi-tarea alternando entre tareas."""
        if verbose:
            print(f"=== Multi-Task Learning ({self.n_tasks} tareas) ===\n")
        
        losses_por_tarea = [[] for _ in range(self.n_tasks)]
        
        for epoch in range(epochs):
            # Alternar entre tareas
            for task_id, (X, y) in enumerate(tareas_datos):
                # Mini-batch
                idx = np.random.choice(len(y), min(32, len(y)), replace=False)
                X_b, y_b = X[idx], y[idx]
                
                # Forward
                probs = self.forward(X_b, task_id)
                n_c = probs.shape[1]
                loss = -np.mean(np.log(probs[np.arange(len(y_b)), y_b] + 1e-10))
                losses_por_tarea[task_id].append(loss)
                
                # Backward (simplificado: solo actualizar con numerical gradient parcial)
                y_oh = np.zeros((len(y_b), n_c))
                y_oh[np.arange(len(y_b)), y_b] = 1
                delta = (probs - y_oh) / len(y_b)
                
                # Gradientes del task head
                pesos_t = self.task_pesos[task_id]
                biases_t = self.task_biases[task_id]
                for i in range(len(pesos_t) - 1, -1, -1):
                    dW = self.task_acts[i].T @ delta
                    db = delta.sum(axis=0)
                    if i > 0:
                        delta = (delta @ pesos_t[i].T) * (self.task_acts[i] > 0)
                    pesos_t[i] -= lr * dW
                    biases_t[i] -= lr * db
                
                # Propagar gradiente al backbone compartido
                delta_shared = delta @ pesos_t[0].T * (self.shared_acts[-1] > 0) if len(pesos_t) > 0 else delta
                
                for i in range(len(self.shared_pesos) - 1, -1, -1):
                    dW = self.shared_acts[i].T @ delta_shared
                    db = delta_shared.sum(axis=0)
                    if i > 0:
                        delta_shared = (delta_shared @ self.shared_pesos[i].T) * (self.shared_acts[i] > 0)
                    self.shared_pesos[i] -= lr * dW
                    self.shared_biases[i] -= lr * db
            
            if verbose and (epoch + 1) % 50 == 0:
                accs = []
                for tid, (X, y_data) in enumerate(tareas_datos):
                    pred = np.argmax(self.forward(X, tid), axis=1)
                    accs.append(np.mean(pred == y_data))
                print(f"  Epoch {epoch+1}: " + ", ".join(f"T{i}={a:.3f}" for i, a in enumerate(accs)))
        
        return losses_por_tarea

# --- Crear 3 tareas relacionadas ---
np.random.seed(42)

# Tarea 0: fácil, muchos datos
X0, y0 = make_classification(n_samples=500, n_features=10, n_informative=8,
                              n_classes=3, random_state=42)
X0 = StandardScaler().fit_transform(X0)

# Tarea 1: media, pocos datos
X1, y1 = make_classification(n_samples=100, n_features=10, n_informative=8,
                              n_classes=3, random_state=43)
X1 = StandardScaler().fit_transform(X1)

# Tarea 2: difícil, pocos datos
X2, y2 = make_classification(n_samples=80, n_features=10, n_informative=5,
                              n_classes=3, class_sep=0.8, random_state=44)
X2 = StandardScaler().fit_transform(X2)

tareas = [(X0, y0), (X1, y1), (X2, y2)]

# Multi-Task
mtl = MultiTaskNet(input_dim=10, shared_dims=[64, 32], 
                   task_dims_list=[[16, 3], [16, 3], [16, 3]])
losses_mtl = mtl.entrenar(tareas, epochs=200, lr=0.01)

# Comparar con Single-Task
print("\n=== Comparación: Multi-Task vs Single-Task ===\n")
for tid, (X, y_data) in enumerate(tareas):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y_data, test_size=0.3, random_state=42)
    
    # Single task
    stl = RedNeuronal([10, 64, 32, 16, 3])
    stl.entrenar(X_tr, y_tr, epochs=200, lr=0.01)
    acc_stl = stl.accuracy(X_te, y_te)
    
    # Multi task
    acc_mtl = np.mean(np.argmax(mtl.forward(X_te, tid), axis=1) == y_te)
    
    print(f"  Tarea {tid} (n={len(y_data)}): Single={acc_stl:.4f}, Multi={acc_mtl:.4f}  "
          f"{'✅ MTL mejor' if acc_mtl > acc_stl else '⚠️  STL mejor'}")

## 8. Transfer Learning Negativo

### ¿Cuándo el transfer learning EMPEORA?

**Negative Transfer** ocurre cuando transferir conocimiento de la tarea fuente **perjudica** el rendimiento en la tarea objetivo.

### Causas principales
1. **Dominios demasiado diferentes**: features de ImageNet no ayudan con datos de radar
2. **Tarea fuente demasiado simple**: una tarea trivial no genera features útiles
3. **Over-transfer**: demasiada dependencia de features específicos del dominio fuente
4. **Catastrophic forgetting**: fine-tuning agresivo destruye features útiles

### Cómo detectar y evitar
- **Medir domain distance** antes de transferir (MMD, A-distance)
- **Monitorear validation** durante fine-tuning
- Usar **learning rates pequeños** y **early stopping**
- Comparar siempre contra **baseline sin transfer**

In [ ]:
# ============================================================
# Experimento: Transfer Learning Negativo
# ============================================================

print("=== Transfer Learning: Positivo vs Negativo ===\n")

# Tarea fuente: clasificación estándar
X_fuente, y_fuente = make_classification(n_samples=2000, n_features=20,
                                          n_informative=15, n_classes=5, random_state=42)
X_fuente = StandardScaler().fit_transform(X_fuente)

# Pre-entrenar
red_pre = RedNeuronal([20, 128, 64, 32, 5])
red_pre.entrenar(X_fuente, y_fuente, epochs=200, lr=0.01)

# Tarea objetivo similar (transfer positivo esperado)
X_similar, y_similar = make_classification(n_samples=100, n_features=20,
                                            n_informative=12, n_classes=3, random_state=100)
X_similar = StandardScaler().fit_transform(X_similar)

# Tarea objetivo muy diferente (transfer negativo posible)
# Datos con estructura completamente diferente
np.random.seed(42)
X_diferente = np.random.randn(100, 20)
# Clase basada en interacciones no lineales complejas
y_diferente = ((np.sin(X_diferente[:, 0] * X_diferente[:, 1]) + 
                np.cos(X_diferente[:, 2] ** 2) + 
                X_diferente[:, 3] * X_diferente[:, 4]) > 0).astype(int)
y_diferente = np.where(X_diferente[:, 5] > 0.5, 2, y_diferente)  # 3 clases

resultados_transfer = {}

for nombre_tarea, X_obj, y_obj in [('Similar', X_similar, y_similar),
                                     ('Muy diferente', X_diferente, y_diferente)]:
    X_tr, X_te, y_tr, y_te = train_test_split(X_obj, y_obj, test_size=0.3, random_state=42)
    n_clases = len(np.unique(y_obj))
    
    # Desde cero
    red_scratch = RedNeuronal([20, 128, 64, 32, n_clases])
    red_scratch.entrenar(X_tr, y_tr, epochs=200, lr=0.01)
    acc_scratch = red_scratch.accuracy(X_te, y_te)
    
    # Con transfer
    red_transfer = RedNeuronal([20, 128, 64, 32, n_clases])
    red_transfer.copiar_pesos_de(red_pre, hasta_capa=3)
    red_transfer.entrenar(X_tr, y_tr, epochs=200, lr=0.001)
    acc_transfer = red_transfer.accuracy(X_te, y_te)
    
    diff = acc_transfer - acc_scratch
    tipo = "✅ POSITIVO" if diff > 0 else "❌ NEGATIVO"
    
    resultados_transfer[nombre_tarea] = {
        'scratch': acc_scratch,
        'transfer': acc_transfer,
        'diff': diff,
        'tipo': tipo
    }
    
    # Medir distancia de dominio
    mmd_val = mmd_rbf(X_fuente[:200], X_obj, gamma=0.1)
    
    print(f"Tarea '{nombre_tarea}' (MMD={mmd_val:.4f}):")
    print(f"  Desde cero: {acc_scratch:.4f}")
    print(f"  Transfer:   {acc_transfer:.4f}  ({'+' if diff > 0 else ''}{diff:.4f}) → {tipo}")
    print()

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Barras comparativas
ax = axes[0]
x = np.arange(len(resultados_transfer))
width = 0.35
nombres_t = list(resultados_transfer.keys())
scratch_vals = [resultados_transfer[n]['scratch'] for n in nombres_t]
transfer_vals = [resultados_transfer[n]['transfer'] for n in nombres_t]

bars1 = ax.bar(x - width/2, scratch_vals, width, label='Desde cero', color='lightcoral', edgecolor='black')
bars2 = ax.bar(x + width/2, transfer_vals, width, label='Transfer', color='steelblue', edgecolor='black')

ax.set_xticks(x); ax.set_xticklabels(nombres_t, fontsize=11)
ax.set_ylabel('Accuracy'); ax.set_title('Transfer Positivo vs Negativo', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

# Anotar diferencias
for i, n in enumerate(nombres_t):
    diff = resultados_transfer[n]['diff']
    color = 'green' if diff > 0 else 'red'
    ax.annotate(f'{"+"+str(round(diff,3)) if diff > 0 else round(diff,3)}',
               xy=(i + width/2, transfer_vals[i] + 0.02),
               fontsize=11, fontweight='bold', color=color, ha='center')

# 2. Guía de decisión
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')

texto = ("GUIA: Cuando hacer Transfer Learning?

"
        "SI transferir cuando:
"
        "  - Dominios similares (MMD bajo)
"
        "  - Pocos datos en la tarea objetivo
"
        "  - La tarea fuente es mas compleja
"
        "  - Features de bajo nivel son compartidos

"
        "NO transferir cuando:
"
        "  - Dominios muy diferentes (MMD alto)
"
        "  - Suficientes datos en la tarea objetivo
"
        "  - Tarea fuente demasiado simple
"
        "  - La estructura del problema es diferente

"
        "SIEMPRE comparar contra baseline
"
        "sin transfer learning")

ax.text(0.05, 0.95, texto, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax.set_title('Guía de Decisión', fontweight='bold')

plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Experimento Completo: Pipeline de Transfer Learning end-to-end
# ============================================================

print("=" * 65)
print("  EXPERIMENTO COMPLETO: Transfer Learning en Dígitos")
print("=" * 65)

# Dataset: Digits (8x8 = 64 features)
digits = load_digits()
X_all = digits.data / 16.0
y_all = digits.target

# Tarea fuente: dígitos 0-7 (muchos datos)
mask_fuente = y_all <= 7
X_fuente_d = X_all[mask_fuente]
y_fuente_d = y_all[mask_fuente]

# Tarea objetivo: dígitos 8-9 (pocos datos simulados)
mask_obj = y_all >= 8
X_obj_d = X_all[mask_obj]
y_obj_d = (y_all[mask_obj] - 8).astype(int)  # 0=ocho, 1=nueve

# Simular "pocos datos" tomando subsets
resultados_por_n = {}

for n_objetivo in [10, 20, 50, 100, len(X_obj_d)]:
    n_real = min(n_objetivo, len(X_obj_d))
    idx = np.random.choice(len(X_obj_d), n_real, replace=False)
    X_o = X_obj_d[idx]
    y_o = y_obj_d[idx]
    X_tr, X_te, y_tr, y_te = train_test_split(X_o, y_o, test_size=0.3, random_state=42)
    
    if len(X_tr) < 2 or len(np.unique(y_tr)) < 2:
        continue
    
    # Pre-entrenar en tarea fuente
    red_pre_d = RedNeuronal([64, 128, 64, 32, 8])
    red_pre_d.entrenar(X_fuente_d, y_fuente_d, epochs=150, lr=0.01)
    
    accs = {}
    
    # 1. Desde cero
    red_s = RedNeuronal([64, 128, 64, 32, 2])
    red_s.entrenar(X_tr, y_tr, epochs=200, lr=0.01)
    accs['Desde cero'] = red_s.accuracy(X_te, y_te)
    
    # 2. Feature Extraction
    feat_tr = red_pre_d.extraer_features(X_tr, capa=3)
    feat_te = red_pre_d.extraer_features(X_te, capa=3)
    clf = LogisticRegression(max_iter=500).fit(feat_tr, y_tr)
    accs['Feature Extraction'] = clf.score(feat_te, y_te)
    
    # 3. Fine-tune completo
    red_ft = RedNeuronal([64, 128, 64, 32, 2])
    red_ft.copiar_pesos_de(red_pre_d, hasta_capa=3)
    red_ft.entrenar(X_tr, y_tr, epochs=200, lr=0.001)
    accs['Fine-tune'] = red_ft.accuracy(X_te, y_te)
    
    # 4. Fine-tune con capas congeladas
    red_fc = RedNeuronal([64, 128, 64, 32, 2])
    red_fc.copiar_pesos_de(red_pre_d, hasta_capa=3)
    red_fc.congelar_capas([0, 1])  # Congelar primeras 2
    red_fc.entrenar(X_tr, y_tr, epochs=200, lr=0.005)
    accs['Freeze + FT'] = red_fc.accuracy(X_te, y_te)
    
    # 5. KNN baseline
    knn = KNeighborsClassifier(3).fit(X_tr, y_tr)
    accs['KNN baseline'] = knn.score(X_te, y_te)
    
    resultados_por_n[n_real] = accs

# Mostrar resultados
print(f"\n{'N datos':>8s}", end='')
for metodo in ['Desde cero', 'Feature Extraction', 'Fine-tune', 'Freeze + FT', 'KNN baseline']:
    print(f"  {metodo:>18s}", end='')
print()
print("-" * 105)

for n, accs in sorted(resultados_por_n.items()):
    print(f"{n:8d}", end='')
    mejor = max(accs.values())
    for metodo in ['Desde cero', 'Feature Extraction', 'Fine-tune', 'Freeze + FT', 'KNN baseline']:
        v = accs.get(metodo, 0)
        marker = " ★" if v == mejor else "  "
        print(f"  {v:16.4f}{marker}", end='')
    print()

# Gráfico
fig, ax = plt.subplots(figsize=(12, 6))
ns = sorted(resultados_por_n.keys())
metodos = ['Desde cero', 'Feature Extraction', 'Fine-tune', 'Freeze + FT', 'KNN baseline']
colores = ['lightcoral', 'skyblue', 'steelblue', 'navy', 'gray']
markers = ['o', 's', '^', 'D', 'x']

for metodo, color, marker in zip(metodos, colores, markers):
    vals = [resultados_por_n[n].get(metodo, 0) for n in ns]
    ax.plot(ns, vals, f'-', color=color, marker=marker, lw=2, markersize=8, label=metodo)

ax.set_xlabel('Número de ejemplos en tarea objetivo', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Transfer Learning: Accuracy vs Tamaño del Dataset Objetivo\n'
             '(Pre-entrenado en dígitos 0-7, evaluado en 8-9)', fontweight='bold', fontsize=13)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.set_xscale('log')

plt.tight_layout(); plt.show()
print("\n✅ Transfer learning es más valioso con MENOS datos")
print("   Con suficientes datos, entrenar desde cero se acerca o supera al transfer")

## Resumen: Transfer Learning

### Mapa conceptual

```
                    TRANSFER LEARNING
                    "Reutilizar conocimiento"
                           │
        ┌──────────────────┼──────────────────┐
        │                  │                  │
   FEATURE             FINE-TUNING       DOMAIN
   EXTRACTION                           ADAPTATION
        │                  │                  │
   Congelar red      Adaptar pesos      Alinear
   + clasificador    pre-entrenados     distribuciones
   nuevo             (LR bajo)         (CORAL, DANN)
        │                  │
        │          ┌───────┼───────┐
        │          │       │       │
        │      Última   Completo  Gradual
        │      capa               Unfreezing
        │
   ┌────┴────────────────────────────┐
   │      OTROS PARADIGMAS           │
   │                                 │
   │  Knowledge Distillation         │
   │  (teacher → student)            │
   │                                 │
   │  Multi-Task Learning            │
   │  (backbone compartido)          │
   └─────────────────────────────────┘
```

### Lo que implementamos desde cero

| Componente | Clase | Técnica clave |
|------------|-------|---------------|
| Red modular | `RedNeuronal` | Congelamiento por capa, extracción de features |
| Domain Shift | `mmd_rbf`, `proxy_a_distance` | Medir distancia entre dominios |
| Feature Extraction | `TransferLearner` | Capas congeladas + clasificador |
| Fine-Tuning | `TransferLearner` | Última capa, completo, gradual, discriminative LR |
| CORAL | `CORAL` | Alineamiento de covarianza |
| DANN | `DomainAdversarial` | Gradient reversal layer |
| Knowledge Distillation | `KnowledgeDistiller` | Soft labels + temperatura |
| Multi-Task | `MultiTaskNet` | Backbone compartido + task heads |

### Reglas prácticas

| Datos objetivo | Similitud dominio | Estrategia |
|---------------|-------------------|-----------|
| **Pocos** | **Alta** | Feature Extraction |
| **Pocos** | **Baja** | Fine-tune últimas capas |
| **Muchos** | **Alta** | Fine-tune completo (LR bajo) |
| **Muchos** | **Baja** | Domain Adaptation o desde cero |